# Baseline Modeling

Purpose: build a fold-safe baseline, save OOF/test predictions, and produce the first valid `submission.csv`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

SEED = 42
N_SPLITS = 5
DATA_DIR = Path('/kaggle/input/playground-series-s6e7')
if not DATA_DIR.exists():
    DATA_DIR = Path('../data')
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('..')
PRED_DIR = OUTPUT_DIR / 'predictions'
PRED_DIR.mkdir(exist_ok=True)

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

id_col = sample_submission.columns[0]
target_candidates = [col for col in train.columns if col not in test.columns]
target = target_candidates[0]

feature_cols = [col for col in test.columns if col in train.columns and col != id_col]
X = train[feature_cols]
y = train[target]
X_test = test[feature_cols]

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [col for col in feature_cols if col not in num_cols]
classes = np.array(sorted(y.unique()))

print('target:', target)
print('classes:', classes)
print('numeric:', len(num_cols), 'categorical:', len(cat_cols))

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), num_cols),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
            ]),
            cat_cols,
        ),
    ],
    remainder='drop',
)

model = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=160,
    l2_regularization=0.05,
    early_stopping=True,
    random_state=SEED,
)

pipeline = Pipeline([
    ('preprocess', preprocess),
    ('model', model),
])

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_pred = pd.Series(index=train.index, dtype=object)
test_votes = []
fold_rows = []

for fold, (trn_idx, val_idx) in enumerate(cv.split(X, y), start=1):
    X_trn, X_val = X.iloc[trn_idx], X.iloc[val_idx]
    y_trn, y_val = y.iloc[trn_idx], y.iloc[val_idx]
    pipeline.fit(X_trn, y_trn)
    val_pred = pipeline.predict(X_val)
    oof_pred.iloc[val_idx] = val_pred
    test_votes.append(pipeline.predict(X_test))
    fold_rows.append({
        'fold': fold,
        'accuracy': accuracy_score(y_val, val_pred),
        'balanced_accuracy': balanced_accuracy_score(y_val, val_pred),
        'macro_f1': f1_score(y_val, val_pred, average='macro'),
    })

results = pd.DataFrame(fold_rows)
display(results)
display(results.mean(numeric_only=True).to_frame('mean'))

pd.DataFrame({'id': train[id_col] if id_col in train else train.index, 'oof_pred': oof_pred}).to_csv(
    PRED_DIR / 'hgb_oof_predictions.csv', index=False
)

In [ ]:
vote_frame = pd.DataFrame(np.column_stack(test_votes))
test_pred = vote_frame.mode(axis=1)[0]

submission = sample_submission.copy()
submission.iloc[:, 1] = test_pred.values
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
display(submission.head())
display(submission.iloc[:, 1].value_counts(normalize=True).to_frame('share'))